# Ground truth extraction

Following up on the station census in `notebook 01`, this notebook pulls the monthly averaged PM2.5 data from all the 
stations identified previously.

In [1]:
%load_ext autoreload
%autoreload 2

## Imports

In [2]:
from pathlib import Path

import dotenv
import pickle
import gzip
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import yaml

from openaq import OpenAQ

import monitoring_deserts.data.fetch_openaq as fetch_oaq

## Configuration

Read the configuration from the previous step, produced in `notebook 01`.

In [3]:
# Run-specific settings
with open("../data/raw/station_census/census_scope.yaml") as f:
    run_config = yaml.safe_load(f)
    
BOUNDING_BOX = tuple(run_config["bounding_box"].values())
DAY_START, DAY_END = run_config["period"]["start"], run_config["period"]["end"]
PM25_PARAM_ID = run_config["pm25_parameter_id"]


# Generic settings
with open("../config.yaml") as f:
    gen_config = yaml.safe_load(f)
    
RAW_DATA_DIR = Path(gen_config["data_dir"]["raw"]) / "ground_truth"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DATA_DIR = Path(gen_config["data_dir"]["clean"]) / "ground_truth"
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)

dotenv.load_dotenv()
oaq_client = OpenAQ()

## Step 1
Retrieve the stations and active sensors list from the previous notebook, and build a list to query OpenAQ.

In [4]:
# The previous nb produced a stations_active.csv file
active_df = pd.read_csv(RAW_DATA_DIR / "../station_census/stations_active.csv")

# Clean the df
columns = { "id" : "station_id",
            "country.name" : "country_name",
            "coordinates.longitude" : "longitude",
            "coordinates.latitude" : "latitude",
            "pollutant_sensor_id" : "pm25_sensor_id",
          }
active_df = active_df[list(columns.keys())].rename(columns=columns)


## Step 2
Query OpenAQ pulling the measurements for the active PM2.5 sensors. Since the OpenAQ API request limit is roughly
1 request per second, pulling these data takes long. So for later executions of this notebook should comment out the
pull request and use the saved data instead. The resulting dataframe is also saved as .parquet and should be the default
object used for subsequent analysis.

In [18]:
# This request might take up to 20 minutes: run it only once,
# all the following notebook executions can use the saved data

#measurement_list = fetch_oaq.pull_measurements(oaq_client, active_df["pm25_sensor_id"],)
## Pickle and compress the raw data for later use
#with gzip.open(RAW_DATA_DIR / "measurement_list.pkl.gz", "wb") as gf:
#    pickle.dump(measurement_list, gf)                                          
## DataFrame with the raw data and save it as a parquet file for later use too
#measurement_df = fetch_oaq.concat_measurements(measurement_list)
#measurement_df.to_parquet(RAW_DATA_DIR / "measurement_df.parquet")

Pulling sensor 50 of 930...
Pulling sensor 100 of 930...
Pulling sensor 150 of 930...
Pulling sensor 200 of 930...
Pulling sensor 250 of 930...
Pulling sensor 300 of 930...
Pulling sensor 350 of 930...
Pulling sensor 400 of 930...
Pulling sensor 450 of 930...
Pulling sensor 500 of 930...
Pulling sensor 550 of 930...
Pulling sensor 600 of 930...
Pulling sensor 650 of 930...
Pulling sensor 700 of 930...
Pulling sensor 750 of 930...
Pulling sensor 800 of 930...
Pulling sensor 850 of 930...
Pulling sensor 900 of 930...


In [16]:
## Uncompress and unpickle the raw data insted of redownloading it
with gzip.open(RAW_DATA_DIR / "measurement_list.pkl.gz", "rb") as gf:
    measurement_list = pickle.load(gf)    
## Read the measurement DataFrame
measurement_df = pd.read_parquet(RAW_DATA_DIR / "measurement_df.parquet")

In [19]:
measurement_df.head()

concentration   unit percent_coverage
sensor_id time_utc                                                       
12234787  2025-01-31 18:30:00+00:00          81.0  µg/m³             36.0
          2025-02-28 18:30:00+00:00          75.4  µg/m³             97.0
          2025-03-31 18:30:00+00:00          84.2  µg/m³             93.0
          2025-04-30 18:30:00+00:00          70.3  µg/m³            100.0
          2025-05-31 18:30:00+00:00          53.0  µg/m³            100.0